# Model Implementation

This script is dedicated to implementing the size invariant saliency prediction model described in our previous presentations, as well as our paper.
It is part of the module "Human-Computer Interfaces" at the university of Leipzig, taught by Patrick Ebel.

The used dataset can be found here: <br>
https://github.com/YueJiang-nj/UEyes-CHI2023

For the dataset we will be using the saliency heatmaps from the folder "heatmaps_1s" and "heatmaps_3s". <br>
(And the basic images from the image folder.)

The implementation is consists of 2 steps:
1. Importing the dataset and building the necessary data loader for training.
2. Building the model itself, defining its architecture as well as hyper- / trainingsparameters.

### 0. Imports

In [ ]:
import os
import timm
import torch
import random
import pandas
import patchify
import numpy as np
from PIL import Image
import torch.nn as nn
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset

### 1. Importing the Data
Loading in all the images from their folders. <br>
Splitting them into a train / test split. <br>
Removing overly large images.

Depending on workflow / speed import could later decide between image subject. <br>
Then only train on certain image types for a start.

In [ ]:
# Define the paths to the dataset:
# (Only works in Jupyter Notebook)
# [ATTENTION]: MODIFY THESE PATHS IF YOUR IMAGES AND TARGETS ARE LOCATED ELSEWHERE
script_directory = os.getcwd()
image_directory = os.path.join(script_directory, "dataset", "images")
target_directory = os.path.join(script_directory, "dataset", "heatmaps_3s")

# Getting the lists for train and test:
# (We decided on a 80/20 train-test-split since the dataset is somewhat small.)
# (So we assume that more training examples will benefit us more than the reduced test variance.)
shuffle_seed = 36
training_ratio = 0.8

# Deterministic shuffling:
image_name_list = os.listdir(image_directory)
random.seed(shuffle_seed)
random.shuffle(image_name_list)

train_test_split = int(training_ratio * len(image_name_list))

train_names = image_name_list[:train_test_split]
test_names = image_name_list[train_test_split:]

In [ ]:
def load_images(file_path, file_names):
# The function tries to load all images in the list "file_names" from the folder "file_path".

    image_set = []
    
    for name in file_names:
        image_path = os.path.join(file_path, name)
        loaded_image = Image.open(image_path)
        image_set.append(loaded_image)

    return image_set

In [ ]:
# Load the different sets of data:
train_images = load_images(image_directory, train_names)
train_targets = load_images(target_directory, train_names)
print("Finished loading the training sets.")

test_images = load_images(image_directory, test_names)
test_targets = load_images(target_directory, test_names)
print("Finished loading the test sets.")

In [ ]:
# Optional tests on data loading:
print(len(train_images)) # Should be 1584
print(len(test_images)) # Should be 396

print(type(train_images[0])) # Should be PIL Image
print(train_images[0].size) # Should be (1890, 1080)
print(train_images[0].mode) # Should be RGB

#train_images[0].show()

In [ ]:
def remove_large_images(image_list, pixel_limit):
# The function detects images with heights / widths over the pixel limit.
# It returns the reduced image list and prints the number of images removed.
    removed_images = 0
    filtered_list = []
    
    for image in image_list:

        if image.size[0] >= pixel_limit or image.size[1] >= pixel_limit:

            removed_images += 1

        else:

            filtered_list.append(image)

    print(f"A total of {removed_images} images have been removed.")
    return filtered_list

In [ ]:
# Very large images blow up calculation time due to drastically heightened patch-count.
# Since the number of images of this type in the dataset is negligeble we decided to remove them.
pixel_limit = 3000

train_images = remove_large_images(train_images, pixel_limit)
train_targets = remove_large_images(train_targets, pixel_limit)

test_images = remove_large_images(test_images, pixel_limit)
test_targets = remove_large_images(test_targets, pixel_limit)

### 1.1 Pre-Processing
The Dataset / Dataloader will be provided with the imported images and is supposed to feed them to the model during training. <br>
The "\_\_getitem\_\_" method contains the pre-processing of images as well as targets. <br>

Pre-Processing contains the following steps:
- Padding
- Patching
- Flattening\*
- Normalization\*\*
- Positional Encoding

\* Original dimensions of the picture will be saved. <br>
\*\* Only done on the training images.

#### 1.1.1 Padding:

In [ ]:
def pad_image(input_image, patch_size):
# The function pads an image with zeros if the image dimension is not perfectly divisible by patch size.

    input_image = np.array(input_image)
    image_dimensions = input_image.shape
    image_height = image_dimensions[0]
    image_width = image_dimensions[1]

    height_overhang = image_height  % patch_size
    width_overhang = image_width % patch_size

    # If it is a colour image:
    if len(image_dimensions) == 3:
            
        # If there is overhang:
        if height_overhang != 0:
    
            # Add padding:
            padding_amount = patch_size - height_overhang
            input_image = np.pad(input_image, pad_width=((0, padding_amount), (0, 0), (0, 0)), mode='constant', constant_values=0)
    
        # If there is overhang:
        if width_overhang != 0:
    
            #Add padding:
            padding_amount = patch_size - width_overhang
            input_image = np.pad(input_image, pad_width=((0, 0), (0, padding_amount), (0, 0)), mode='constant', constant_values=0)
            
        return input_image

    # If it is a black and white image:
    # (Or just an image with more or less than 3 channels)
    else:
    
        # If there is overhang:
        if height_overhang != 0:
    
            # Add padding:
            padding_amount = patch_size - height_overhang
            input_image = np.pad(input_image, pad_width=((0, padding_amount), (0, 0)), mode='constant', constant_values=0)
    
        # If there is overhang:
        if width_overhang != 0:
    
            #Add padding:
            padding_amount = patch_size - width_overhang
            input_image = np.pad(input_image, pad_width=((0, 0), (0, padding_amount)), mode='constant', constant_values=0)

        return input_image

In [ ]:
# Optional Test:
print(np.array(train_images[0]).shape) # Should be (1080, 1890, 3)
print(pad_image(train_images[0], 32).shape) # Should be (1088, 1920, 3)

#### 1.1.2 Patching:

In [ ]:
def create_patches(padded_image, patch_size):
# The function cuts an image into square patches of the provided size.
    
# Info: Currently step size if coupled to patch size.
# Therfore we currently have no overlap between patches.
    image_dimensions = padded_image.shape

    # If it is a colour image:
    if len(image_dimensions) == 3:

        patched_image = patchify.patchify(padded_image,(patch_size, patch_size, padded_image.shape[2]), step = patch_size)
        patched_image = np.squeeze(patched_image)
        # [IMPORTANT:] The squeeze has to be undone in order to reconstruct the image

    # If it is a black and white image:
    # (Or just an image with more or less than 3 channels)
    else:

        patched_image = patchify.patchify(padded_image,(patch_size, patch_size), step = patch_size)

    return patched_image

In [ ]:
# Optional Test:
padded_image = pad_image(train_images[0], 32)
patched_image = create_patches(padded_image, 32)
print(padded_image.shape) # Should be (1088, 1920, 3)
print(patched_image.shape) # Should be (34, 60, 32, 32, 3)
print(f"Original height is {patched_image.shape[0] * 32}")
print(f"Original height is {patched_image.shape[1] * 32}")

#### 1.1.3 Flatten + Normalization + Tensor conversion

In [ ]:
def flatten_matrix(patch_matrix, patch_size):
# The function flattens the matrix, turns it into a tensor, and normalises it, based on image type.

    matrix_dimensions = patch_matrix.shape
    row_number = matrix_dimensions[0]
    column_number = matrix_dimensions[1]
    
    if len(matrix_dimensions) == 5:
    # If the image is RGB:
        
        flattened_matrix = patch_matrix.reshape(-1, patch_size, patch_size, 3)
        flattened_matrix = torch.tensor(flattened_matrix, dtype=torch.float32).permute(0, 3, 1, 2)
        # Flatten and reshape
        
        normalize = transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
        flattened_matrix = normalize(flattened_matrix / 255.0)
        # Normalize
        
        return flattened_matrix, row_number, column_number
    
    elif len(matrix_dimensions) == 4:
    # If the image is one-dimensional:
    
        flattened_matrix = patch_matrix.reshape(-1, patch_size, patch_size)
        flattened_matrix = flattened_matrix.reshape(flattened_matrix.shape[0], -1)
        flattened_matrix = torch.tensor(flattened_matrix, dtype=torch.float32)
        # Flatten and reshape
        
        return flattened_matrix, row_number, column_number
    
    else:
    # Unknown input:
    
        print("The input does not conform to the expected dimensions.\nPlease only enter RGB or black-and-white images!")
        return

In [ ]:
# Optional Test:

# Test on colour
padded_image = pad_image(train_images[0], 32)
patched_image = create_patches(padded_image, 32)
flattened_image, image_height, image_width = flatten_matrix(patched_image, 32)

# Test on black and white
padded_target = pad_image(train_targets[0], 32)
patched_target = create_patches(padded_target, 32)
flattened_target, target_height, target_width = flatten_matrix(patched_target, 32)

print(flattened_image.shape) # Should be [2040, 3, 32, 32]
print(flattened_target.shape) # Should be [2040, 32, 32]

#### 1.1.4 Generation of the positional encoding:

In [ ]:
def create_positional_encoding(row_number, column_number):
# This function creates a numpy array containing the relative postions of every patch.

    encoding_matrix = np.zeros((row_number, column_number, 2))
    
    for row in range(row_number):
        for column in range(column_number):
    
            height_encoding = row / (row_number - 1)
            width_encoding = column / (column_number - 1)
            # -1 because the rows are indices while the number is an absolute count.
            # Otherwise the last element would only get a positional value close to 1 while it should be 1.
    
            encoding_matrix[row, column, 0] = height_encoding
            encoding_matrix[row, column, 1] = width_encoding

    flattened_encoding = encoding_matrix.reshape(-1, 2)
    flattened_encoding = torch.tensor(flattened_encoding, dtype=torch.float32)
    
    return flattened_encoding

In [ ]:
# Optional Test:
padded_image = pad_image(train_images[0], 32)
patched_image = create_patches(padded_image, 32)
flattened_image, image_height, image_width = flatten_matrix(patched_image, 32)
positional_encoding = create_positional_encoding(image_height, image_width)

print(positional_encoding.shape) # Should be (2040, 2)

#### 1.1.5 Pre-Processing function:

In [ ]:
def preprocess_image(image, patch_size, calculate_positional, is_input = True):
# This function applies padding, patching, flattening, normalization and tensor conversion.
# Additionally it creates a positional encoding if it is needed in the pipeline.

    # Ensure format is RGB:
    if is_input == True:
        image = image.convert("RGB")

    # Ensure format is RGB:
    else:
        image = image.convert("L")
        
    padded_image = pad_image(image, patch_size)
    patched_image = create_patches(padded_image, patch_size)
    flattened_image, image_height, image_width = flatten_matrix(patched_image, patch_size)

    if calculate_positional == True:
        positional_encoding = create_positional_encoding(image_height, image_width)
        return flattened_image, positional_encoding
        
    else:
        return flattened_image

In [ ]:
# Optional Test:
processed_image, positional_encoding = preprocess_image(train_images[0], 32, True, True)
processed_target = preprocess_image(train_targets[0], 32, False, False)

print(type(positional_encoding)) # Has to be <class 'torch.Tensor'>
print(positional_encoding.shape) # Should be [2040, 2]

print(type(processed_image)) # Has to be <class 'torch.Tensor'>
print(processed_image.shape) # Should be [2040, 3, 32, 32]

print(type(processed_target)) # Has to be <class 'torch.Tensor'>
print(processed_target.shape) # Should be [2040, 1024]

### 1.2 Dataset definition

In [ ]:
class PatchedDataset(Dataset):
    def __init__(self, images, targets, transform = None, patch_size = 32):
        self.images = images
        self.targets = targets
        self.transform = transform
        self.patch_size = patch_size

    def __len__(self):
        return len(self.images)

    def __getitem__(self,idx):
        x = self.images[idx]
        y = self.targets[idx]

        # Application of the pre-processing:
        x, positional_encoding = self.transform(x, self.patch_size, True, True)
        y = self.transform(y, self.patch_size, False, False)

        return x, y, positional_encoding

## 2. Model creation and training
The model itself consists of three major parts:

- a CNN encoder to extract features from the patches
- a transformer encoder to apply cross attention to the patches
- a linear layer that works as a decoder to produce a flattened saliency patch

Between the CNN-Encoder and the transformer encoder a positional encoding is added to each patch.
Furthermore a global token is appended to gather global information.
A projection layer between the CNN and transformer ensures correct sizes.

In [ ]:
# Make sure a graphics card is utilized if present:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"You are training on: {device}.")

#### 2.1 Define the model:

In [ ]:
class PatchPredictor(nn.Module):
    
    def __init__(self, in_channels = 3, patch_size = 32, device = "cuda"):
        super().__init__()
        
        self.in_channels = in_channels
        self.patch_size = patch_size
        self.device = device

        # Construct the CNN-encoder:
        self.cnn_encoder = nn.Sequential(  
            nn.Conv2d(in_channels, 32, kernel_size = 3, padding = 1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2), # This leaves us with a 16x16 image with 32 channels.

            nn.Conv2d(32, 48, kernel_size=3, padding=1),
            nn.BatchNorm2d(48),
            nn.ReLU(),
            nn.MaxPool2d(2),  # This leaves us with a 8x8 images with 48 channels. 

            nn.Conv2d(48, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(2), # 2 * 2 value in each of the 64 channels.

            nn.Flatten()
        )

        # Construct the projection layer
        self.projection_layer = nn.Linear(258,384)

        # Construct a layer of the transformer encoder:
        transformer_layer = nn.TransformerEncoderLayer(
            d_model = 384,
            nhead = 8,
            dim_feedforward = 512,
            dropout = 0.1,
            activation = 'relu'
        )

        # Construct the full length transformer and normalization layer:
        self.transformer = nn.TransformerEncoder(transformer_layer, num_layers = 3)
        self.norm = nn.LayerNorm(384)

        # Construct final decoder layer
        self.decoder = nn.Sequential(
            nn.Linear(770, 768),
            nn.ReLU(),
            nn.Linear(768,1024) 
        )

        # Preallocate cls_token as a buffer on GPU
        cls_token_init = torch.zeros(384)
        cls_token_init[-2:] = 0.5
        self.register_buffer('cls_token', cls_token_init)
        
    def forward(self, x, positional_encodings):
    # Note: This structure is not compatible with batching as the different numbers of patches per image cant be stacked into a rectengular tensor.
    # You could work around this behaviour by a) concatinating batches with patches -> you gotta restore the original patch structures later.
        
        # Run the feature extraction
        x = self.cnn_encoder(x)

        # Attach the positional encodings and project to tranformer size
        x = torch.cat([x, positional_encodings], dim = 1)
        x = self.projection_layer(x)

        # Create and attach the global token  
        batch_cls = self.cls_token.unsqueeze(0)
        x = torch.cat([x, batch_cls], dim = 0)

        # Use cross attention on the projected patch features
        # (Transposition would only be necessary if batch size differs from 1)
        #x = x.transpose(0, 1)       
        x = self.transformer(x)           
        x = self.norm(x)
        #x = x.transpose(0, 1) 

        # Save and remove the global token, give it the size of x
        cls_token = x[-1, :].unsqueeze(0).expand(x.size(0) - 1, -1)
        x = x[:-1, :]

        # Add positional encoding and cls token again, for clarity
        x = torch.cat([x, cls_token, positional_encodings], dim = 1) # Transformer size * 2 + 2

        # Let the decoder predict the patch
        x = self.decoder(x)
        
        return x

#### 2.2 Define Training and Validation

In [ ]:
# Initialize the dataloader and model:
train_dataloader = PatchedDataset(train_images, train_targets, preprocess_image, 32)
test_dataloader = PatchedDataset(test_images, test_targets, preprocess_image, 32)
patch_saliency = PatchPredictor(device = device)
patch_saliency = patch_saliency.to(device)

# Adjust the optimizer:
learning_rate = 5e-5
weight_decay = 1e-3
optimizer = torch.optim.AdamW(patch_saliency.parameters(), lr = learning_rate, weight_decay = weight_decay)

# Define the loss function:
# (An alternative maybe: mse_loss + 0.1 * l1_loss)
criterion = nn.MSELoss()

# Define training for a single epoch:
def train_one_epoch(dataset, epoch, path, report_on_step = 100):

    running_loss = 0
    loss_since_report = 0
    step_number = 0
    patch_saliency.train()
    
    for batch in dataset:
    # Define training for a single batch:
        
        # Define the data variables
        training_data = batch[0]
        eval_data = batch[1]
        positional_enc = batch[2]
        
        training_data = training_data.to(device)
        eval_data = eval_data.to(device)
        positional_enc = positional_enc.to(device)

        # Run the model
        output = patch_saliency(training_data, positional_enc)
        
        # Calculate loss and gradient
        loss = criterion(output, eval_data)
        optimizer.zero_grad()
        loss.backward()
        
        # Adjust model weights
        optimizer.step()

        # Generate report data (error per patch instead of per image):
        running_loss += loss.item() / training_data.shape[0]
        loss_since_report += loss.item() / training_data.shape[0] 
        step_number += 1

        # Periodically report to console after a certain amount of batches:
        if step_number % report_on_step == 0:

            # Empty the cache to prevent slowdown
            torch.cuda.empty_cache()

            # Log the results
            status_report = f"| EPOCH: {epoch} | Step {step_number} | Current loss: {loss.item():.4f} | Average patch-loss: {loss_since_report / report_on_step:.4f} |\n"
            print(status_report)
            with open(path, "a") as f:
                f.write(status_report)

            # Resetten des Report-Loses
            loss_since_report = 0

    torch.cuda.empty_cache()
    average_loss = running_loss / step_number
    print(f"Average loss for the epoch: {average_loss:.4f}")

    return average_loss

# Define validation function:
def validate_model(dataset):

    # Set model to evaluation mode
    running_validation_loss = 0
    patch_saliency.eval()
    step_number = 0

    for batch in dataset:
    # Define validation behaviour for each batch

        # Define data variables
        input_data = batch[0]
        eval_data = batch[1]
        positional_enc = batch[2]

        input_data = input_data.to(device)
        eval_data = eval_data.to(device)
        positional_enc = positional_enc.to(device)
        
        # Run the model
        with torch.no_grad():
            output = patch_saliency(input_data, positional_enc)
        
        # Generate report data (error per patch instead of per image):
        loss = criterion(output, eval_data)
        running_validation_loss += loss.item() / input_data.shape[0]
        step_number += 1

    average_validation_loss = running_validation_loss / step_number
    print(f"Average validation patch-loss for the epoch is: {average_validation_loss:.4f}")
          
    return average_validation_loss  

In [ ]:
# Run the full training for the specified number of epochs:
epoch_number = 10

# [ATTENTION]: MODIFY THIS NAME FOR EACH NEW MODEL OTHERWISE EXISTING LOGS AND CHECKPOINTS WILL BE OVERWRITTEN
model_name = "rich_context_3s"

# [ATTENTION]: MODIFY THESE PATHS IF YOUR SAVED LOGS AND CHECKPOINTS SHOULD BE WRITTEN TO ANOTHER LOCATION
log_file = os.path.join(script_directory, "models", "logs", model_name + ".txt")
checkpoint_file = os.path.join(script_directory, "models", "checkpoints")

training_loss = []
validation_loss = []

# Report on step should at max be 200 as the report is coupled to the emptying of the cache.
# If the cache is not emptied often enough the model will slow down significantly.
for epoch in range(1, epoch_number + 1):
    
    train_loss = train_one_epoch(train_dataloader, epoch = epoch, path = log_file, report_on_step = 50)
    val_loss = validate_model(test_dataloader)

    training_loss.append(train_loss)
    validation_loss.append(val_loss)

    # Add the final training and validation loss to the logs:
    with open(log_file, "a") as f:
        f.write(" ".join(map(str, training_loss)) + "\n")
        f.write(" ".join(map(str, validation_loss)) + "\n")

    # Save a torch-checkpoint of the model:
    torch.save({
    "epoch": epoch,
    "model_state_dict": patch_saliency.state_dict(),
    "optimizer_state_dict": optimizer.state_dict()
    }, os.path.join(checkpoint_file, f"{model_name}_e{epoch}.pt"))